# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution: Exploration with `mlcroissant`

This notebook provides a step-by-step template for loading and exploring the [FAIR^2 dataset](https://sen.science/doi/10.71728/senscience.qs2f-h81p) using the [`mlcroissant`](https://github.com/mlcommons/croissant) library, which enables programmatic access to public datasets described in the [MLCommons Croissant](https://mlcommons.github.io/croissant/) schema.

### Dataset Source
The dataset source is provided via a Croissant schema JSON-LD URL:

```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```

We'll refer to all dataset entities (record sets, fields, etc.) by their `@id`.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant

## 1. Data Loading

Load the dataset, including metadata and available records, from the Croissant schema using `mlcroissant`. We also display metadata summary to understand what the dataset contains.

In [ ]:
import mlcroissant as mlc
import pandas as pd
from pprint import pprint

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access the metadata (always as a single object!)
metadata = dataset.metadata
print(f"Dataset: {metadata.name}\n")
print(f"Identifier: {metadata.identifier}")
print(f"Description: {metadata.description}\n")
print(f"License: {metadata.license}")
print(f"Date Published: {metadata.datePublished}")
print(f"Keywords: {getattr(metadata, 'keywords', None)}")


## 2. Data Overview

Review the dataset's available record sets, fields, and corresponding `@id` values. All exploration and extraction in this notebook will refer to these `@id`s.

Let's inspect all record sets, their fields, and field types. This will help us choose which data to extract and analyze. Note that you must use the `@id` when accessing data programmatically with `mlcroissant`.

In [ ]:
# List all record sets and their @id
record_sets = list(dataset.record_sets)
print(f"Found {len(record_sets)} record sets:")
for rs in record_sets:
    print(f"  - {rs['@id']}: {rs['name'] if 'name' in rs else rs.get('@id', '')}")

# Show the fields and their types for each record set
for rs in record_sets:
    print(f"\nRecord Set '@id': {rs['@id']}")
    fields = rs.get('field', [])
    if isinstance(fields, dict): # single field
        fields = [fields]
    if not fields:
        print("  [No fields defined]")
        continue
    for field in fields:
        field_id = field.get('@id', str(field)) if isinstance(field, dict) else str(field)
        name = field.get('name', field_id) if isinstance(field, dict) else field_id
        dtype = field.get('dataType', '') if isinstance(field, dict) else ''
        print(f"  - Field '@id': {field_id} | Name: {name} | Type: {dtype}")

## 3. Data Extraction

We'll load data from a specific record set using its `@id`, as found above. The records will be loaded into a DataFrame. You can adapt this to extract from any desired record set by changing the `record_set_id` variable.

In [ ]:
# -- extract all record sets into DataFrames --
# Replace the following with the @id(s) found in Data Overview (Section 2) as appropriate.

target_record_sets = [rs['@id'] for rs in record_sets]
dataframes = {}

for record_set_id in target_record_sets:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            dataframes[record_set_id] = pd.DataFrame(records)
            print(f"Loaded {len(records)} records from record set: {record_set_id}")
        else:
            print(f"No records found in record set: {record_set_id}")
    except Exception as e:
        print(f"Could not load record set {record_set_id}: {e}")

# Pick a non-empty record set for further exploration
for rs_id, df in dataframes.items():
    if not df.empty:
        selected_record_set_id = rs_id
        break
else:
    selected_record_set_id = None

if selected_record_set_id:
    print(f"\nFirst non-empty record set chosen for EDA: {selected_record_set_id}\n")
    print("Field IDs (column names):")
    print(list(dataframes[selected_record_set_id].columns))
    display(dataframes[selected_record_set_id].head())
else:
    print("No records loaded from any record set.")

## 4. Exploratory Data Analysis (EDA)

Let's apply some common data processing steps on a selected (non-empty) record set DataFrame using field `@id`s (column headers).

* We'll select a numeric field (by `@id`) for analysis, e.g., filter records above a threshold.
* Normalize the numeric field.
* Group by a relevant categorical field, e.g., MSI status or sex.

> **Reminder**: You must use the field `@id` strings as columns in the DataFrame. Adjust the `numeric_field_id` and `group_field_id` below according to what's actually present in your DataFrame.

In [ ]:
"""
Replace the values of `numeric_field_id` and `group_field_id` with actual column `@id`s from your dataset. 
The code below demonstrates a typical EDA pipeline.
"""
# Example setup for EDA
df = dataframes[selected_record_set_id]

# Display all columns to find a suitable numeric field and group field
print("Columns (field @id):", list(df.columns))

# Choose two available @id column names (see the output above) that make sense numerically and categorically.
numeric_field_id = df.select_dtypes(include=['number', 'float64', 'int64']).columns[0] if not df.select_dtypes(include=['number', 'float64', 'int64']).empty else df.columns[0]
group_field_id = df.columns[1] if len(df.columns) > 1 else numeric_field_id

print(f"\nUsing field '@id' for numeric analysis: {numeric_field_id}")
print(f"Using field '@id' for grouping: {group_field_id}")

# Filter for values greater than a threshold
threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 10
filtered_df = df[df[numeric_field_id] > threshold]
print(f"\nFiltered records with {numeric_field_id} > {threshold}:")
display(filtered_df.head())

# Normalize the field
if pd.api.types.is_numeric_dtype(filtered_df[numeric_field_id]):
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    )
    print(f"\nNormalized '{numeric_field_id}' for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Grouping and aggregation
if group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"\nGrouped mean of '{numeric_field_id}' by '{group_field_id}':")
    display(grouped_df.head())

## 5. Visualization

Let's visualize the distribution of the selected numeric field and its relation to a categorical group, using the field `@id`s.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot the distribution
plt.figure(figsize=(8, 4))
sns.histplot(df[numeric_field_id], kde=True)
plt.title(f"Distribution of field {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# Boxplot by group, if available
if group_field_id in df.columns:
    plt.figure(figsize=(8, 4))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion

- We have successfully loaded and explored the [FAIR^2 CRC](https://sen.science/doi/10.71728/senscience.qs2f-h81p) Croissant dataset using `mlcroissant`.
- All entities (record sets, fields, etc.) were referenced and accessed strictly via their Croissant `@id`.
- The notebook demonstrates loading, field and schema inspection, data extraction, basic analysis and visualization.
- You can extend this workflow for more nuanced analysis and/or integrate additional ML/data science pipelines as needed.

***
For further details, visit the [MLCommons Croissant documentation](https://github.com/mlcommons/croissant).